# SI7003 NLP - SI7016 Applied NLP
#
# Lecture03 - Momento 2
#
## Modelos de Lenguaje n-gram con KenLM: *scoring*, autocompletado, generación y perplejidad

**Objetivo:** construir y usar un LM n‑gram como *sistema*.

> Notebook demo, mini-retos y un gran reto al final.

---


## 0. Requisitos y notas
- Este notebook usa **KenLM**.

In [ ]:
!rm -rf kenlm

In [ ]:
# Instalar KenLM en la máquina local, tiene que ser un ubuntu o leer las instrucciones especificas para Mac o Windows
# en caso de no tener nativamente linux, puede instalar docker y correr ubuntu.
# se adjuntan los Dockerfile y docker-compose para esto
#
# este demo tambien corre en google colab
#
!apt-get update
!apt-get install -y build-essential cmake libboost-all-dev zlib1g-dev
!git clone https://github.com/kpu/kenlm.git
!cd kenlm && mkdir build && cd build && cmake .. && make -j4

In [ ]:
# === Configuración básica ===
import os, re, math, random, subprocess, sys
from pathlib import Path

SEED = 42
random.seed(SEED)

DATA_DIR = Path("data")
MODELS_DIR = Path("models")
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("MODELS_DIR:", MODELS_DIR.resolve())


In [ ]:
!pip3 install kenlm

In [ ]:
# === 1.1 Instalar/Importar kenlm ===
try:
    import kenlm
    print("kenlm ya está disponible.")
except Exception:
    print("Instalando kenlm...")
    !pip3 install kenlm
    import kenlm
    print("kenlm instalado.")


## 2. Corpus para demo (AG News)
Construimos un corpus simple concatenando el campo `text`.


In [ ]:
# === Descargar AG News (Hugging Face datasets) ===
try:
    from datasets import load_dataset
except Exception:
    !pip -q install datasets
    from datasets import load_dataset

ds = load_dataset("ag_news")
print(ds)

N_TRAIN = 20000  # trae en total 120.000, configura el valor a trabajar.
train_lines = [(ex["text"] or "").replace("\n", " ").strip() for ex in ds["train"].select(range(N_TRAIN))]

corpus_path = DATA_DIR / "agnews_dataset.txt"
corpus_path.write_text("\n".join(train_lines), encoding="utf-8")
print("Corpus:", corpus_path, "| líneas:", len(train_lines))


### 2.2 Normalización mínima
Para LM no quitamos stopwords. Hacemos limpieza ligera.


In [ ]:
# === Normalizar ===
raw = corpus_path.read_text(encoding="utf-8").splitlines()

def normalize(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

norm_lines = [normalize(x) for x in raw if x.strip()]
norm_path = DATA_DIR / "agnews_dataset.norm.txt"
norm_path.write_text("\n".join(norm_lines), encoding="utf-8")

print("Normalizado:", norm_path, "| líneas:", len(norm_lines))
print("Ejemplo:", norm_lines[0][:200], "...")


### Verificar binarios KenLM (lmplz/build_binary)
En algunos entornos `kenlm` python existe pero `lmplz` no.


In [ ]:
# === 2.4 Localizar binarios ===
import shutil
def which(cmd): return shutil.which(cmd)

print("lmplz:", which("lmplz"))
print("build_binary:", which("build_binary"))


## 3. Entrenamiento rápido (n=2 a n=5, variando según el caso)
Usamos Kneser-Ney (por defecto `lmplz`).


In [ ]:
# === Funciones entrenamiento ===
def train_kenlm_ngram(corpus_txt: Path, n: int, out_arpa: Path):
    cmd = f'kenlm/build/bin/lmplz -o {n} --discount_fallback < "{corpus_txt}" > "{out_arpa}"'
    print("CMD:", cmd)
    subprocess.run(cmd, shell=True, check=True)

def build_binary(arpa: Path, out_bin: Path):
    cmd = f'kenlm/build/bin/build_binary "{arpa}" "{out_bin}"'
    print("CMD:", cmd)
    subprocess.run(cmd, shell=True, check=True)

arpa2 = MODELS_DIR / "agnews_2.arpa"
bin2  = MODELS_DIR / "agnews_2.binary"
arpa3 = MODELS_DIR / "agnews_3.arpa"
bin3  = MODELS_DIR / "agnews_3.binary"
arpa4 = MODELS_DIR / "agnews_4.arpa"
bin4  = MODELS_DIR / "agnews_4.binary"
arpa5 = MODELS_DIR / "agnews_5.arpa"
bin5  = MODELS_DIR / "agnews_5.binary"

have_bins = (which("lmplz") is not None) and (which("build_binary") is not None)
print("¿Binarios disponibles?", have_bins)

have_bins = True


In [ ]:
# === Entrenar LM n-gram con diferentes n, puede seleccionar con cual quiere trabajar ===
# Si es lento: entrena solo n=3 (trigram).
if have_bins:
    if not bin2.exists():
        train_kenlm_ngram(norm_path, 2, arpa2)
        build_binary(arpa2, bin2)
    if not bin3.exists():
        train_kenlm_ngram(norm_path, 3, arpa3)
        build_binary(arpa3, bin3)
    if not bin4.exists():
        train_kenlm_ngram(norm_path, 4, arpa4)
        build_binary(arpa4, bin4)
    if not bin5.exists():
        train_kenlm_ngram(norm_path, 5, arpa5)
        build_binary(arpa5, bin5)

for p in [bin2, bin3, bin4, bin5]:
    print(p.name, "exists?", p.exists(), "size(MB):", (p.stat().st_size/1e6 if p.exists() else None))


In [ ]:
# === Asegurar un modelo ===
bins = list(MODELS_DIR.glob("*.binary"))
print("Modelos encontrados:", [b.name for b in bins])
assert len(bins) > 0, "No hay modelos .binary. Entrena o copia uno a models/."


## 4. Cargar modelo y hacer *sentence scoring*


In [ ]:
# === Cargar modelo ===
import kenlm
# elija con cual n quiere trabajar
model_path = bin3 if bin3.exists() else bins[0]
model = kenlm.Model(str(model_path))
print("Modelo cargado:", model_path)


In [ ]:
# === Scoring ===
def score_sentence(m: kenlm.Model, s: str) -> float:
    s = normalize(s)
    return m.score(s, bos=True, eos=True)

tests = [
    "today is tuesday",
    "today tuesday is",
    "global warming effects",
    "warming global effects",
    "machine learning models",
    "learning machine models",
]
for t in tests:
    print(f"{t:28s}  score={score_sentence(model, t):.3f}")


### Mini‑reto
Crea 3 variantes de una frase (cambiando el orden) y compara scores. Explica qué captura el modelo.


## 5. Ejemplo — Detector de rarezas


In [ ]:
# === Umbral de rareza ===
def is_rare(m: kenlm.Model, s: str, threshold: float) -> bool:
    return score_sentence(m, s) < threshold

thr = -20.0  # ajusta

candidates = [
    "the stock market fell today",
    "market stock the fell today",
    "quantum banana economics",
    "economic report shows inflation",
]

for s in candidates:
    sc = score_sentence(model, s)
    print(f"{sc:8.2f}  rare={sc < thr}  | {s}")


### Mini‑reto
Ajusta `thr` para que marque como raras ~2 frases. ¿Por qué un umbral fijo falla entre dominios?


## 6. Ejemplo — Autocompletado Top‑k (aproximado)
Construimos un vocabulario candidato (top palabras del corpus) y rankeamos.


In [ ]:
# === Vocabulario candidato ===
from collections import Counter

words = Counter()
for line in norm_lines[:5000]:
    words.update(line.split())

VOCAB = [w for w,_ in words.most_common(500)]
print("Tamaño vocab candidato:", len(VOCAB))
print("Top-20:", VOCAB[:20])


In [ ]:
# === Autocompletar ===
def autocomplete_topk(m: kenlm.Model, context: str, vocab, k: int = 5):
    context = normalize(context).strip()
    scores = []
    for w in vocab:
        s = f"{context} {w}".strip()
        scores.append((score_sentence(m, s), w))
    scores.sort(reverse=True, key=lambda x: x[0])
    return scores[:k]

for ctx in ["the president of", "stock market", "new york", "artificial intelligence"]:
    print("\nContexto:", ctx)
    for sc, w in autocomplete_topk(model, ctx, VOCAB, k=5):
        print(f"  {w:18s}  score={sc:.2f}")


### Mini‑reto 3
Cambia el tamaño de vocab candidato (100, 500, 2000). ¿Cambian los top‑k? ¿Por qué depende tanto del vocabulario?


## 7. Ejemplo — Generación (greedy vs sampling)


In [ ]:
# === Generación greedy ===
def generate_greedy(m: kenlm.Model, start: str = "the", max_len: int = 25):
    tokens = normalize(start).split()
    for _ in range(max_len - len(tokens)):
        ctx = " ".join(tokens[-2:]) if len(tokens) >= 2 else " ".join(tokens)
        next_w = autocomplete_topk(m, ctx, VOCAB, k=1)[0][1]
        tokens.append(next_w)
    return " ".join(tokens)

print(generate_greedy(model, start="the", max_len=25))


In [ ]:
# === Generación con sampling top-k ===
def generate_sample(m: kenlm.Model, start: str = "the", max_len: int = 25, k: int = 30, temperature: float = 1.0):
    tokens = normalize(start).split()
    for _ in range(max_len - len(tokens)):
        ctx = " ".join(tokens[-2:]) if len(tokens) >= 2 else " ".join(tokens)
        top = autocomplete_topk(m, ctx, VOCAB, k=k)
        scs = [sc for sc,_ in top]
        mx = max(scs)
        weights = [math.exp((sc - mx)/max(1e-6, temperature)) for sc,_ in top]
        s = sum(weights)
        probs = [w/s for w in weights]
        r = random.random()
        cum = 0.0
        chosen = top[-1][1]
        for p, (_, w) in zip(probs, top):
            cum += p
            if r <= cum:
                chosen = w
                break
        tokens.append(chosen)
    return " ".join(tokens)

for T in [0.5, 1.0, 1.5]:
    print("\nT=", T, "=>", generate_sample(model, start="the", max_len=25, k=30, temperature=T))


### Micro‑reto
Genera 3 textos con greedy y 3 con sampling. Identifica repetición y plantillas. ¿Qué limitación del n‑gram lo explica?


## 8. Perplejidad (evaluación intrínseca)


In [ ]:
# === Split train/test ===
lines = norm_path.read_text(encoding="utf-8").splitlines()
random.shuffle(lines)
split = int(0.9 * len(lines))
train_txt = DATA_DIR / "train.txt"
test_txt  = DATA_DIR / "test.txt"
train_txt.write_text("\n".join(lines[:split]), encoding="utf-8")
test_txt.write_text("\n".join(lines[split:]), encoding="utf-8")
print("train:", split, "test:", len(lines)-split)


In [ ]:
# === Perplejidad con KenLM ===
pp = model.perplexity(test_txt.read_text(encoding="utf-8"))
print("Perplexity (test):", pp)


### Micro‑reto
Si puedes re‑entrenar con menos/más datos (`N_TRAIN`), compara perplejidad. ¿Por qué más datos suele bajar perplejidad? ¿Cuándo no?


## 9. Gran Reto de clase
Hacer un clasificador basado en los datos de entrenamiento y pruebas de AG_NEWS, aprovechando que cada texto tiene su etiqueta, la estrategia de elaboración de este Gran Reto es el siguiente:
- Separar el dataset de entrenamiento, en n sub-dataset, cada uno por label.
- Entrenar y crear n modelos LM n-gram con la librería KenLM (gran pregunta: **¿Qué n utilizar? n=2,3,4,5**
- crear los binarios de cada uno de los modelos.
- para un subconjunto de datos (x,y) del conjunto de pruebas (test) de AG_NEWS, tomar cada texto (x) y pasarlo por cada uno de los modelos entrenados, asignar el label y' al que le de, más baja perplejidad.
- comparar y vs y'
- realizar las métricas de precision, recall, f1-score y concluir.
- reentrenar con diferentes tamaños de n en el modelo lm n-gram, y hallar el n óptimo. Que métrica nos daría ese 'optimo'